# SKL metric analysis

## Step 1 — Read training and metric results

- `read_training_results(...)` reads the last epoch row from each training log.
- `read_metric_results(...)` reads the first `BeforeRescale` row from each metric log.

Both functions return one flat `pandas.DataFrame` row per `.txt` file.

In [33]:
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
from mpl_toolkits.axes_grid1 import make_axes_locatable
from sklearn.linear_model import LinearRegression
from adjustText import adjust_text

from matplotlib import colormaps
from matplotlib.colors import LinearSegmentedColormap

### Filename parsing

The patterns below define the hyperparameters encoded in the filenames. Add another `(column_name, pattern)` entry if a future experiment introduces a new filename field.

In [34]:
NUMBER_PATTERN = r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?"

# Each regex captures only the value following a known filename prefix.
HYPERPARAM_PATTERNS = (
    ("aug", rf"(?:^|_)aug(?P<value>True|False)(?=_|$)"),
    ("disableNorm", rf"(?:^|_)disableNorm(?P<value>True|False)(?=_|$)"),
    ("opt", r"(?:^|_)opt(?P<value>[^_]+)(?=_|$)"),
    ("epochs", rf"(?:^|_)epochs(?P<value>\d+)(?=_|$)"),
    ("bsize", rf"(?:^|_)bsize(?P<value>\d+)(?=_|$)"),
    ("LR", rf"(?:^|_)LR(?P<value>{NUMBER_PATTERN})(?=_|$)"),
    # ("PWD", rf"(?:^|_)PWD(?P<value>{NUMBER_PATTERN})(?=_|$)"),
    ("WD", rf"(?:^|_)WD(?P<value>{NUMBER_PATTERN})(?=_|$)"),
    # ("FuncWD", rf"(?:^|_)FuncWD(?P<value>{NUMBER_PATTERN})(?=_|$)"),
    ("mom", rf"(?:^|_)mom(?P<value>{NUMBER_PATTERN})(?=_|$)"),
    # ("NJ", rf"(?:^|_)NJ(?P<value>{NUMBER_PATTERN})(?=_|$)"),
    ### `scheduler` and `plateau` share one filename token in current logs.
    # ("scheduler", r"(?:^|_)scheduler(?P<value>True|False)"),
    # ("plateau", rf"plateau(?P<value>{NUMBER_PATTERN})(?=_|$)"),
    # ("rho", rf"(?:^|_)rho(?P<value>{NUMBER_PATTERN})(?=_|$)"),
    # ("adapsam", r"(?:^|_)adapsam(?P<value>True|False)(?=_|$)"),
    # ("labelsm", rf"(?:^|_)labelsm(?P<value>{NUMBER_PATTERN})(?=_|$)"),
)

HYPERPARAM_COLUMNS = [name for name, _ in HYPERPARAM_PATTERNS]

PERFORMANCE_COLUMNS = [
    "epoch_runned",
    "train_loss",
    "train_acc",
    "test_loss",
    "test_acc",
]


def smart_cast(value: str):
    """Convert filename values to bool/int/float when possible."""
    value = value.strip()
    if value in {"True", "False"}:
        return value == "True"
    if re.fullmatch(r"[-+]?\d+", value):
        return int(value)
    if re.fullmatch(NUMBER_PATTERN, value):
        return float(value)
    return value


def parse_hparams_from_filename(log_path):
    """Extract known hyperparameter choices from a training or metric filename."""
    stem = Path(log_path).stem
    hparams = {}
    for column, pattern in HYPERPARAM_PATTERNS:
        match = re.search(pattern, stem)
        if match is not None:
            hparams[column] = smart_cast(match.group("value"))
    return hparams

### Shared log helpers

In [35]:
EPOCH_PATTERN = re.compile(r"^Epoch\s+(\d+)")
EPOCHS_RUN_PATTERN = re.compile(r"^--Epochs run:\s*(\d+)")


def parse_numeric(value: str) -> float:
    """Parse log numbers, including scientific notation, nan/inf, and times ending in `s`."""
    value = value.strip()
    if value.endswith("s"):
        value = value[:-1].strip()
    try:
        return float(value)
    except ValueError as exc:
        raise ValueError(f"Expected a numeric log value, got {value!r}") from exc


def parse_epoch(line: str) -> int:
    match = EPOCH_PATTERN.search(line.strip())
    if match is None:
        raise ValueError(f"Could not parse an epoch from: {line!r}")
    return int(match.group(1))


def parse_pipe_fields(line: str, start_at: int = 1) -> dict:
    """Parse `label: value` fields separated by vertical bars."""
    fields = {}
    for part in line.split("|")[start_at:]:
        part = part.strip()
        if not part or ":" not in part:
            continue
        label, value = part.split(":", 1)
        fields[label.strip()] = parse_numeric(value)
    return fields


def ordered_dataframe(rows: list[dict], leading_columns: list[str]) -> pd.DataFrame:
    """Put identifiers/hyperparameters first and retain every discovered result column."""
    frame = pd.DataFrame(rows)
    leading = [column for column in leading_columns if column in frame.columns]
    remaining = [column for column in frame.columns if column not in leading]
    return frame.loc[:, leading + remaining]


def sort_result_dataframe(frame, sorting_key, sorting_order):
    """Sort result rows by one column using an explicit order."""
    order_to_ascending = {"ascending": True, "descending": False}
    if sorting_order not in order_to_ascending:
        raise ValueError(
            "sorting_order must be either 'ascending' or 'descending'"
        )
    if sorting_key not in frame.columns:
        raise KeyError(
            f"Sorting key {sorting_key!r} is not a result column"
        )

    return frame.sort_values(
        by=sorting_key,
        ascending=order_to_ascending[sorting_order],
        na_position="last",
        kind="stable",
    ).reset_index(drop=True)


def collect_result_rows(log_dir, parse_file, pattern="*.txt", strict=False):
    """Apply a single-file parser to all matching files in a directory."""
    log_dir = Path(log_dir)
    files = sorted(log_dir.glob(pattern))
    if not files:
        raise FileNotFoundError(f"No files matching {pattern!r} in {log_dir}")

    rows = []
    for filepath in files:
        try:
            rows.append(parse_file(filepath))
        except (OSError, ValueError, KeyError) as exc:
            if strict:
                raise RuntimeError(f"Failed to parse {filepath}") from exc
            warnings.warn(f"Skipping {filepath}: {exc}", stacklevel=2)
    if not rows:
        raise RuntimeError(f"No valid result rows were parsed from {log_dir}")
    return rows

### Training-result reader

Performance loss/scc values and final_LR come from the last Epoch ... row. 

epoch_runned uses the explicit --Epochs run: footer when present; otherwise it is the zero-based last epoch index plus one.

In [36]:
TRAINING_FIELD_MAP = {
    "LR": "final_LR",
    "Train Loss": "train_loss",
    "Train Acc": "train_acc",
    "Test Loss": "test_loss",
    "Test Acc": "test_acc",
}


def parse_training_result_file(log_path) -> dict:
    """Parse one training log into one flat result dictionary."""
    log_path = Path(log_path)
    last_epoch_line = None
    epochs_runned = None

    with log_path.open("r", encoding="utf-8") as handle:
        for raw_line in handle:
            line = raw_line.strip()
            if EPOCH_PATTERN.match(line):
                last_epoch_line = line
            footer_match = EPOCHS_RUN_PATTERN.match(line)
            if footer_match is not None:
                epochs_runned = int(footer_match.group(1))

    if last_epoch_line is None:
        raise ValueError("No epoch logging row was found")

    fields = parse_pipe_fields(last_epoch_line)
    missing = [label for label in TRAINING_FIELD_MAP if label not in fields]
    if missing:
        raise ValueError(f"Last epoch row is missing fields: {missing}")

    # Epoch indices are zero-based, while epoch_runned is a count.
    if epochs_runned is None:
        epochs_runned = parse_epoch(last_epoch_line) + 1

    row = {"filepath": str(log_path)}
    row.update(parse_hparams_from_filename(log_path))
    row["epoch_runned"] = epochs_runned
    for log_label, column in TRAINING_FIELD_MAP.items():
        row[column] = fields[log_label]
    return row


def read_training_results(
    log_dir, pattern="*.txt", strict=False,
    sorting_key="test_loss", sorting_order="descending",
) -> pd.DataFrame:
    """Read and sort all training logs into one row-per-file DataFrame."""
    rows = collect_result_rows(
        log_dir, parse_training_result_file, pattern=pattern, strict=strict
    )
    leading = ["filepath", 
               *HYPERPARAM_COLUMNS, 
               *PERFORMANCE_COLUMNS, 
               "final_LR"]
    frame = ordered_dataframe(rows, leading)
    return sort_result_dataframe(frame, sorting_key, sorting_order)

### Metric-result reader

By default, read `newtrained33*.txt` files and use only the first `BeforeRescale` row.

The five requested performance fields receive standardized names; 

Every other labeled value retains its exact log label, including all 15 `structureS_sig1..5_alpha1..3` grid entries and `inputS_alpha1..2`.

Consequently, a field such as 'KL_norm_uniform: nan' still creates a 'KL_norm_uniform' column containing 'NaN'.

In [37]:
METRIC_PERFORMANCE_MAP = {
    "Train Loss": "train_loss",
    "Train Acc": "train_acc",
    "Test Loss": "test_loss",
    "Test Acc": "test_acc",
}


def parse_metric_result_file(log_path) -> dict:
    """Parse the first BeforeRescale row in one metric-result log."""
    log_path = Path(log_path)
    before_rescale_line = None

    with log_path.open("r", encoding="utf-8") as handle:
        for raw_line in handle:
            line = raw_line.strip()
            parts = [part.strip() for part in line.split("|")]
            if EPOCH_PATTERN.match(line) and len(parts) > 1 and parts[1] == "BeforeRescale":
                before_rescale_line = line
                break

    if before_rescale_line is None:
        raise ValueError("No BeforeRescale row was found")

    # Skip both the epoch and status fields; remaining fields are label/value pairs.
    fields = parse_pipe_fields(before_rescale_line, start_at=2)
    missing = [label for label in METRIC_PERFORMANCE_MAP if label not in fields]
    if missing:
        raise ValueError(f"BeforeRescale row is missing fields: {missing}")

    row = {"filepath": str(log_path)}
    row.update(parse_hparams_from_filename(log_path))
    row["epoch_runned"] = parse_epoch(before_rescale_line)

    # Standardize performance names and preserve every other log label verbatim.
    for log_label, value in fields.items():
        column = METRIC_PERFORMANCE_MAP.get(log_label, log_label)
        row[column] = value
    return row


def read_metric_results(
    log_dir, pattern="newtrained33*.txt", strict=False,
    sorting_key="test_loss", sorting_order="descending",
) -> pd.DataFrame:
    """Read and sort all metric logs into one row-per-file DataFrame."""
    rows = collect_result_rows(
        log_dir, parse_metric_result_file, pattern=pattern, strict=strict
    )
    leading = ["filepath", 
               *HYPERPARAM_COLUMNS, 
               *PERFORMANCE_COLUMNS]
    frame = ordered_dataframe(rows, leading)
    return sort_result_dataframe(frame, sorting_key, sorting_order)

### Load all model results

The path setup works whether Jupyter starts in this notebook's directory or at the workspace root.

In [38]:
cwd = Path.cwd()
if cwd.name == "result-RegressionPCR":
    project_root = cwd.parent
elif (cwd / "result_trainedmodel").is_dir():
    project_root = cwd
elif (cwd / "project_SCI" / "result_trainedmodel").is_dir():
    project_root = cwd / "project_SCI"
else:
    raise FileNotFoundError(
        "Run this notebook from project_SCI, its parent, or result-RegressionPCR."
    )

result_root = project_root / "result_trainedmodel"


# DATA~MODEL
DATASET_MODELS = {
    "cifar10": [
        "resnet18_BN",
        "vgg13_BN",
        "resnet18_noBN",
        "vgg13_noBN",
    ],
    "cifar100": [
        "resnet34_BN",
        "vgg19_BN",
        "wideresnetpostact_BN",
        "vit_BN",
        "wideresnetpostact_noBN",
        "vit_noBN",
    ],
}

# Keep a flat model list for the model-wise loops in later sections.
MODELS = [
    model_name
    for dataset_models in DATASET_MODELS.values()
    for model_name in dataset_models
]

# MODEL~DATA
MODEL_DATASETS = {
    model_name: dataset
    for dataset, dataset_models in DATASET_MODELS.items()
    for model_name in dataset_models
}

SORTING_KEY = "test_loss"
SORTING_ORDER = "descending"

METRIC_PATTERN = "newtrained33*.txt"  #if need sgd- or adamw- specific, filter at here


# Read every directory once and retain dictionaries for convenient iteration.
training_dfs = {}
metric_dfs = {}
load_records = []
for dataset, dataset_models in DATASET_MODELS.items():
    for model_name in dataset_models:
        training_pattern = (
            "*optadamw*.txt"
            if "vit" in model_name.lower()
            else "*.txt" #if need sgd- or adamw- specific, filter at here
        )

        # Training results
        training_dfs[model_name] = read_training_results(
            result_root / dataset / model_name,
            pattern=training_pattern,
            sorting_key=SORTING_KEY,
            sorting_order=SORTING_ORDER,
        )

        # Metric results
        metric_dfs[model_name] = read_metric_results(
            result_root / dataset / f"{model_name}_metrics",
            pattern=METRIC_PATTERN,
            sorting_key=SORTING_KEY,
            sorting_order=SORTING_ORDER,
        )

        load_records.append(
            {
                "dataset": dataset,
                "model": model_name,
                "training_shape": training_dfs[model_name].shape,
                "metric_shape": metric_dfs[model_name].shape,
            }
        )

        print(f"Dataset: {dataset} | Model: {model_name}")

        display(training_dfs[model_name].head(10))
        # display(metric_dfs[model_name].head(5))

load_summary = pd.DataFrame(load_records)
display(load_summary)


# Explicit names make individual model results convenient to use in later cells.
resnet18_BN_training_df = training_dfs["resnet18_BN"]
resnet18_BN_metric_df = metric_dfs["resnet18_BN"]
resnet18_noBN_training_df = training_dfs["resnet18_noBN"]
resnet18_noBN_metric_df = metric_dfs["resnet18_noBN"]

vgg13_BN_training_df = training_dfs["vgg13_BN"]
vgg13_BN_metric_df = metric_dfs["vgg13_BN"]
vgg13_noBN_training_df = training_dfs["vgg13_noBN"]
vgg13_noBN_metric_df = metric_dfs["vgg13_noBN"]

resnet34_BN_training_df = training_dfs["resnet34_BN"]
resnet34_BN_metric_df = metric_dfs["resnet34_BN"]

vgg19_BN_training_df = training_dfs["vgg19_BN"]
vgg19_BN_metric_df = metric_dfs["vgg19_BN"]

wideresnetpostact_BN_training_df = training_dfs["wideresnetpostact_BN"]
wideresnetpostact_BN_metric_df = metric_dfs["wideresnetpostact_BN"]
wideresnetpostact_noBN_training_df = training_dfs["wideresnetpostact_noBN"]
wideresnetpostact_noBN_metric_df = metric_dfs["wideresnetpostact_noBN"]

vit_BN_training_df = training_dfs["vit_BN"]
vit_BN_metric_df = metric_dfs["vit_BN"]
vit_noBN_training_df = training_dfs["vit_noBN"]
vit_noBN_metric_df = metric_dfs["vit_noBN"]

Dataset: cifar10 | Model: resnet18_BN


,filepath,aug,disableNorm,opt,epochs,bsize,LR,WD,mom,epoch_runned,train_loss,train_acc,test_loss,test_acc,final_LR
0,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,sgd,800,128,10000,5000,6,93,2.3026,10.0,2.3026,10.00,0.00010
1,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,sgd,800,128,10000,5000,9,87,2.3026,10.0,2.3026,10.00,0.00010
2,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,sgd,800,128,5000,5000,9,88,2.3026,10.0,2.3026,10.00,0.00005
3,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,adamw,800,128,10000,10,0,216,0.0000,100.0,1.8779,81.79,0.00010
4,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,adamw,800,128,5000,10,0,168,0.0000,100.0,1.6339,83.89,0.00005
5,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,adamw,800,128,5000,50,0,124,0.0000,100.0,1.5877,83.38,0.00005
6,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,adamw,800,128,10000,50,0,180,0.0000,100.0,1.4943,82.63,0.00010
7,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,adamw,800,128,1000,10,0,179,0.0000,100.0,1.4436,85.24,0.00001
8,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,adamw,800,128,10000,100,0,123,0.0000,100.0,1.3880,83.40,0.00010
9,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,adamw,800,128,1000,50,0,148,0.0000,100.0,1.3865,84.95,0.00001


Dataset: cifar10 | Model: vgg13_BN


,filepath,aug,disableNorm,opt,epochs,bsize,LR,WD,mom,epoch_runned,train_loss,train_acc,test_loss,test_acc,final_LR
0,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,adamw,800,128,10000,10,0,287,0.0004,100.00,3.0022,80.58,1.000000e-04
1,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,adamw,800,128,10000,1000,0,800,0.0072,99.89,2.4115,78.86,1.000000e-08
2,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,adamw,800,128,10000,50,0,198,0.0001,100.00,2.4086,82.72,1.000000e-04
3,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,sgd,800,128,10000,5000,6,92,2.3026,10.00,2.3026,10.00,1.000000e-04
4,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,sgd,800,128,10000,5000,9,85,2.3026,10.00,2.3026,10.00,1.000000e-04
5,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,sgd,800,128,5000,5000,9,85,2.3026,10.00,2.3026,10.00,5.000000e-05
6,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,adamw,800,128,10000,100,0,192,0.0000,100.00,1.9956,83.57,1.000000e-04
7,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,adamw,800,128,5000,10,0,170,0.0000,100.00,1.7868,84.71,5.000000e-05
8,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,adamw,800,128,10000,5000,0,800,1.5319,37.06,1.5945,36.40,1.000000e-08
9,/egr/research-pac/chengzi3/project_SCI/result_...,False,False,adamw,800,128,10000,500,0,180,0.0015,99.97,1.5162,84.36,1.000000e-05


Dataset: cifar10 | Model: resnet18_noBN


...

Dataset: cifar100 | Model: wideresnetpostact_noBN


,filepath,aug,disableNorm,opt,epochs,bsize,LR,WD,mom,epoch_runned,train_loss,train_acc,test_loss,test_acc,final_LR
0,/egr/research-pac/chengzi3/project_SCI/result_...,False,True,sgd,800,512,500,1000,9,265,0.0045,99.98,7.8702,30.41,0.000005
1,/egr/research-pac/chengzi3/project_SCI/result_...,False,True,sgd,800,512,1000,1000,9,281,0.0067,99.98,7.0620,32.75,0.000010
2,/egr/research-pac/chengzi3/project_SCI/result_...,False,True,sgd,800,512,1000,1000,6,222,0.0048,99.98,6.5037,35.11,0.000010
3,/egr/research-pac/chengzi3/project_SCI/result_...,False,True,sgd,800,512,1000,500,6,185,0.0018,99.98,6.3472,37.97,0.000010
4,/egr/research-pac/chengzi3/project_SCI/result_...,False,True,sgd,800,128,1000,1000,9,266,0.0071,99.98,5.7339,39.97,0.000010
5,/egr/research-pac/chengzi3/project_SCI/result_...,False,True,sgd,800,512,500,1000,6,223,0.0046,99.98,5.6225,41.28,0.000005
6,/egr/research-pac/chengzi3/project_SCI/result_...,False,True,sgd,800,512,500,500,9,194,0.0025,99.98,5.5315,40.52,0.000005
7,/egr/research-pac/chengzi3/project_SCI/result_...,False,True,sgd,800,128,1000,100,6,142,0.0005,99.98,5.2707,46.78,0.000010
8,/egr/research-pac/chengzi3/project_SCI/result_...,False,True,sgd,800,128,800,100,6,163,0.0005,99.98,5.2562,47.64,0.000008
9,/egr/research-pac/chengzi3/project_SCI/result_...,False,True,sgd,800,128,1000,200,6,139,0.0010,99.98,4.9380,46.99,0.000010


Dataset: cifar100 | Model: vit_noBN


,filepath,aug,disableNorm,opt,epochs,bsize,LR,WD,mom,epoch_runned,train_loss,train_acc,test_loss,test_acc,final_LR
0,/egr/research-pac/chengzi3/project_SCI/result_...,False,True,adamw,800,128,10,50000,0,302,0.0004,99.98,7.2588,34.37,1.953100e-07
1,/egr/research-pac/chengzi3/project_SCI/result_...,False,True,adamw,800,128,10,80000,0,340,0.0005,99.98,7.2515,33.83,1.953100e-07
2,/egr/research-pac/chengzi3/project_SCI/result_...,False,True,adamw,800,128,10,1000,0,322,0.0003,99.98,6.9620,40.64,1.953100e-07
3,/egr/research-pac/chengzi3/project_SCI/result_...,False,True,adamw,800,128,10,10000,0,275,0.0003,99.98,6.9514,38.85,1.953100e-07
4,/egr/research-pac/chengzi3/project_SCI/result_...,False,True,adamw,800,128,10,500,0,288,0.0003,99.98,6.7581,40.36,1.953100e-07
5,/egr/research-pac/chengzi3/project_SCI/result_...,False,True,adamw,800,128,10,8000,0,304,0.0003,99.98,6.7577,40.21,1.953100e-07
6,/egr/research-pac/chengzi3/project_SCI/result_...,False,True,adamw,800,128,10,800,0,306,0.0003,99.98,6.7346,40.40,1.953100e-07
7,/egr/research-pac/chengzi3/project_SCI/result_...,False,True,adamw,800,128,10,5000,0,324,0.0003,99.98,6.6262,40.75,1.953100e-07
8,/egr/research-pac/chengzi3/project_SCI/result_...,False,True,adamw,800,128,20,80000,0,347,0.0004,99.98,6.1688,40.51,3.906300e-07
9,/egr/research-pac/chengzi3/project_SCI/result_...,False,True,adamw,800,128,20,10000,0,280,0.0003,99.98,6.1579,45.06,3.906300e-07


,dataset,model,training_shape,metric_shape
0,cifar10,resnet18_BN,"(72, 15)","(72, 49)"
1,cifar10,vgg13_BN,"(72, 15)","(72, 49)"
2,cifar10,resnet18_noBN,"(60, 15)","(60, 49)"
3,cifar10,vgg13_noBN,"(60, 15)","(60, 49)"
4,cifar100,resnet34_BN,"(48, 15)","(48, 49)"
5,cifar100,vgg19_BN,"(72, 15)","(72, 49)"
6,cifar100,wideresnetpostact_BN,"(48, 15)","(48, 49)"
7,cifar100,vit_BN,"(48, 15)","(48, 49)"
8,cifar100,wideresnetpostact_noBN,"(48, 15)","(48, 49)"
9,cifar100,vit_noBN,"(48, 15)","(48, 49)"


## Observation:
1) more than half of ViT and noLN-ViT trained models severely overfit, with huge test_loss values.

2) noBN-Wideresnet also overfit.

3) VGG19 substantially overfit when opt=adamW

In [39]:
# imp

⬇️------------------
For debug usage. Inpect the dataframe, no need to run this subsection everytime.

In [40]:
# # For Debug
# ACTIVE_MODEL = "wideresnetpostact_noBN"  #"resnet18_BN"

# training_df = training_dfs[ACTIVE_MODEL]
# metric_df = metric_dfs[ACTIVE_MODEL]

# metric_df.columns

# # Inspect only computed metrics, excluding paths, hyperparameters, and performance fields.
# non_metric_columns = {"filepath", *HYPERPARAM_COLUMNS, *PERFORMANCE_COLUMNS}
# metric_columns = [
#     column for column in metric_df.columns
#     if column not in non_metric_columns
# ]

# # Confirm that fields logged as `nan` remain present as DataFrame columns.
# nan_metric_columns = [
#     column for column in metric_columns
#     if metric_df[column].isna().any()]
# print("Metric columns containing at least one NaN:")
# print(nan_metric_columns)

# # Identify metrics that are zero or negative for at least one run.
# nonpositive_metric_columns = [
#     column for column in metric_columns
#     if metric_df[column].le(0).any()]
# print("\nMetric columns containing at least one value <= 0:")
# print(nonpositive_metric_columns)

# # Identify metrics that are strictly negative for at least one run.
# negative_metric_columns = [
#     column for column in metric_columns
#     if metric_df[column].lt(0).any()]
# print("\nMetric columns containing at least one value < 0:")
# print(negative_metric_columns)

⬆️------------------

## Step 2 — Prepare data for analysis

### Step 2.1
### Add derived columns

In [41]:
def add_derived_metric_columns(metric_df):
    """Return a copy with the derived columns used by later analyses."""
    ready_df = metric_df.copy()

    ready_df["gap"] = ready_df["test_loss"] - ready_df["train_loss"]
    ready_df["KL(og)_sqrt"] = np.sqrt(ready_df["KL(og)"])
    return ready_df


# Prepare every model while keeping the parsed metric DataFrames unchanged.
metric_dfs_readytogo = {
    model_name: add_derived_metric_columns(metric_dfs[model_name])
    for model_name in MODELS
}

resnet18_BN_metric_df_readytogo = metric_dfs_readytogo["resnet18_BN"]
resnet18_noBN_metric_df_readytogo = metric_dfs_readytogo["resnet18_noBN"]
vgg13_BN_metric_df_readytogo = metric_dfs_readytogo["vgg13_BN"]
vgg13_noBN_metric_df_readytogo = metric_dfs_readytogo["vgg13_noBN"]


resnet34_BN_metric_df_readytogo = metric_dfs_readytogo["resnet34_BN"]
vgg19_BN_metric_df_readytogo = metric_dfs_readytogo["vgg19_BN"]

wideresnetpostact_BN_metric_df_readytogo = metric_dfs_readytogo["wideresnetpostact_BN"]
wideresnetpostact_noBN_metric_df_readytogo = metric_dfs_readytogo["wideresnetpostact_noBN"]
vit_BN_metric_df_readytogo = metric_dfs_readytogo["vit_BN"]
vit_noBN_metric_df_readytogo = metric_dfs_readytogo["vit_noBN"]

print(
    "Added derived columns: gap and KL(og)_sqrt. "
    "DataFrames 'modelXX_metric_df_readytogo' are ready to go!"
)

Added derived columns: gap and KL(og)_sqrt. DataFrames 'modelXX_metric_df_readytogo' are ready to go!


### Merge single model 'xx_metric_df_readytogo' into 
### Mixed Models

- Cross-arch

resnet18_BN_metric_df + vgg13_BN_metric_df

resnet18_BN_metric_df + vgg13_BN_metric_df + resnet18_noBN_metric_df + vgg13_noBN_metric_df

wideresnetpostact_BN_metric_df + vit_BN_metric_df

wideresnetpostact_BN_metric_df + wideresnetpostact_noBN_metric_df + vit_BN_metric_df + vit_noBN_metric_df


- Cross-arch and Cross-dataset

resnet18_BN_metric_df + vgg13_BN_metric_df + wideresnetpostact_BN_metric_df

resnet18_BN_metric_df + vgg13_BN_metric_df + wideresnetpostact_BN_metric_df + resnet18_noBN_metric_df + vgg13_noBN_metric_df + wideresnetpostact_noBN_metric_df

resnet18_BN_metric_df + vgg13_BN_metric_df + wideresnetpostact_BN_metric_df + vit_BN_metric_df

resnet18_BN_metric_df + vgg13_BN_metric_df + wideresnetpostact_BN_metric_df + vit_BN_metric_df + resnet18_noBN_metric_df + vgg13_noBN_metric_df + wideresnetpostact_noBN_metric_df + vit_noBN_metric_df


In [42]:
def merge_ready_metric_dfs(*ready_dfs):
    """Row-wise merge ready-to-go metric DataFrames with a fresh index."""
    if not ready_dfs:
        raise ValueError("At least one ready-to-go metric DataFrame is required.")
    return pd.concat(ready_dfs, axis=0, ignore_index=True, sort=False)


# Cross-architecture combinations within one dataset group.
resnet18_vgg13_MixedTwo_ready = merge_ready_metric_dfs(
    resnet18_BN_metric_df_readytogo,vgg13_BN_metric_df_readytogo,
)
resnet18_vgg13_MixedFour_ready = merge_ready_metric_dfs(
    resnet18_BN_metric_df_readytogo,vgg13_BN_metric_df_readytogo,
    resnet18_noBN_metric_df_readytogo,vgg13_noBN_metric_df_readytogo,
)

widerenet_vit_MixedTwo_ready = merge_ready_metric_dfs(
    wideresnetpostact_BN_metric_df_readytogo,vit_BN_metric_df_readytogo,
)
widerenet_vit_MixedFour_ready = merge_ready_metric_dfs(
    wideresnetpostact_BN_metric_df_readytogo,wideresnetpostact_noBN_metric_df_readytogo,
    vit_BN_metric_df_readytogo,vit_noBN_metric_df_readytogo,
)


# Cross-architecture and cross-dataset combinations.
resnet18_vgg13_wideresnet_MixedThree_ready = merge_ready_metric_dfs(
    resnet18_BN_metric_df_readytogo,vgg13_BN_metric_df_readytogo,wideresnetpostact_BN_metric_df_readytogo,
)
resnet18_vgg13_wideresnet_MixedSix_ready = merge_ready_metric_dfs(
    resnet18_BN_metric_df_readytogo,vgg13_BN_metric_df_readytogo,wideresnetpostact_BN_metric_df_readytogo,
    resnet18_noBN_metric_df_readytogo,vgg13_noBN_metric_df_readytogo,wideresnetpostact_noBN_metric_df_readytogo,
)

resnet18_vgg13_wideresnet_vit_MixedFour_ready = merge_ready_metric_dfs(
    resnet18_BN_metric_df_readytogo,vgg13_BN_metric_df_readytogo,wideresnetpostact_BN_metric_df_readytogo,vit_BN_metric_df_readytogo,
)
resnet18_vgg13_wideresnet_vit_MixedEight_ready = merge_ready_metric_dfs(
    resnet18_BN_metric_df_readytogo,vgg13_BN_metric_df_readytogo,wideresnetpostact_BN_metric_df_readytogo,vit_BN_metric_df_readytogo,
    resnet18_noBN_metric_df_readytogo,vgg13_noBN_metric_df_readytogo,wideresnetpostact_noBN_metric_df_readytogo,vit_noBN_metric_df_readytogo,
)


mixed_metric_dfs_readytogo = {
    # cross-arch
    "resnet18_vgg13_MixedTwo_ready": resnet18_vgg13_MixedTwo_ready,
    "resnet18_vgg13_MixedFour_ready": resnet18_vgg13_MixedFour_ready,
    "widerenet_vit_MixedTwo_ready": widerenet_vit_MixedTwo_ready,
    "widerenet_vit_MixedFour_ready": widerenet_vit_MixedFour_ready,
    # cross-arch and cros-dataset
    "resnet18_vgg13_wideresnet_MixedThree_ready": resnet18_vgg13_wideresnet_MixedThree_ready,
    "resnet18_vgg13_wideresnet_MixedSix_ready": resnet18_vgg13_wideresnet_MixedSix_ready,
    "resnet18_vgg13_wideresnet_vit_MixedFour_ready": resnet18_vgg13_wideresnet_vit_MixedFour_ready,
    "resnet18_vgg13_wideresnet_vit_MixedEight_ready": resnet18_vgg13_wideresnet_vit_MixedEight_ready,
}

mixed_metric_df_summary = pd.DataFrame(
    [{"mixed_df_name": mixed_name, "shape": mixed_df.shape}
        for mixed_name, mixed_df in mixed_metric_dfs_readytogo.items()
    ]
)
display(mixed_metric_df_summary)

,mixed_df_name,shape
0,resnet18_vgg13_MixedTwo_ready,"(144, 51)"
1,resnet18_vgg13_MixedFour_ready,"(264, 51)"
2,widerenet_vit_MixedTwo_ready,"(96, 51)"
3,widerenet_vit_MixedFour_ready,"(192, 51)"
4,resnet18_vgg13_wideresnet_MixedThree_ready,"(192, 51)"
5,resnet18_vgg13_wideresnet_MixedSix_ready,"(360, 51)"
6,resnet18_vgg13_wideresnet_vit_MixedFour_ready,"(240, 51)"
7,resnet18_vgg13_wideresnet_vit_MixedEight_ready,"(456, 51)"


In [43]:
resnet18_BN_metric_df_readytogo.columns

Index(['filepath', 'aug', 'disableNorm', 'opt', 'epochs', 'bsize', 'LR', 'WD',
       'mom', 'epoch_runned', 'train_loss', 'train_acc', 'test_loss',
       'test_acc', 'S(og)', 'KL(og)', 'KL_spec_norm_prod',
       'KL_spec_norm_prod_n_root', 'KL_path_norm', 'KL_path_norm_n_root',
       'KL_norm_uniform', 'S_trace_uniform', 'KL_norm_minL2', 'S_trace_minL2',
       'S_adap_sam', 'structureS_sig1_alpha1', 'structureS_sig1_alpha2',
       'structureS_sig1_alpha3', 'structureS_sig2_alpha1',
       'structureS_sig2_alpha2', 'structureS_sig2_alpha3',
       'structureS_sig3_alpha1', 'structureS_sig3_alpha2',
       'structureS_sig3_alpha3', 'structureS_sig4_alpha1',
       'structureS_sig4_alpha2', 'structureS_sig4_alpha3',
       'structureS_sig5_alpha1', 'structureS_sig5_alpha2',
       'structureS_sig5_alpha3', 'inputS_alpha1', 'inputS_alpha2',
       'KL_func_det', 'KL_func_det_centered', 'KL_func_det_unsquare_centered',
       'KL_func_det_avg', 'KL_func_det_centered_avg',
       'KL_f

In [44]:
# imp

### Step 2.2 — Define sharpness-complexity metric pairs

`SC_metric_pairs` stores exact DataFrame column names,

`SC_metric_pair_labels` stores labels for PCR plots, and

`single_metrics` stores the unique original column names used later for single-factor regression.

This catalog matches the `newtrained33` schema:

- Structure sharpness: `structureS_sig{sigma}_alpha{alpha}` for sigma indices 1–5 and alpha indices 1–3 (15 combinations).
- Input sharpness: `inputS_alpha1` and `inputS_alpha2`.
- Each structure/input sharpness is paired with `KL_func_det_unsquare_centered`.

`S_Eloss_aniso` is no longer logged, so its pair and plot label are removed.


In [45]:
def build_SC_metric_catalog():
    """Build metric pairs and plot labels for the newtrained33 metric schema."""
    # ===============
    SC_metric_pairs = {
        # "pair_original": ("S(og)", "KL(og)"),
        "pair_original_sqrt": ("S(og)", "KL(og)_sqrt"),
        "pair_adapsam_spec_norm": ("S_adap_sam", "KL_spec_norm_prod"),
        "pair_adapsam_spec_norm_n_root": ("S_adap_sam", "KL_spec_norm_prod_n_root"),
        "pair_adapsam_path_norm": ("S_adap_sam", "KL_path_norm"),
        "pair_adapsam_path_norm_n_root": ("S_adap_sam", "KL_path_norm_n_root"),

        "pair_adapsam_func_det_unsquare_centered": ("S_adap_sam", "KL_func_det_unsquare_centered"),
    }

    sigma_indices = (1, 2, 3, 4, 5)
    alpha_indices = (1, 2, 3)
    input_alpha_indices = (1, 2)

    for sigma in sigma_indices:
        for alpha in alpha_indices:
            pair_name = f"pair_structureS_sig{sigma}_alpha{alpha}_func_det_unsquare_centered"
            structure_sharpness = f"structureS_sig{sigma}_alpha{alpha}"
            SC_metric_pairs[pair_name] = (structure_sharpness, "KL_func_det_unsquare_centered")

    for alpha in input_alpha_indices:
        pair_name = f"pair_inputS_alpha{alpha}_func_det_unsquare_centered"
        input_sharpness = f"inputS_alpha{alpha}"
        SC_metric_pairs[pair_name] = (input_sharpness, "KL_func_det_unsquare_centered")

    metric_plot_labels = {
        "S(og)": "traceH",
        "KL(og)": "l2norm_square",
        "KL(og)_sqrt": "l2norm",
        "S_adap_sam": "adapSAM",
        "KL_spec_norm_prod": "spec_norm_prod",
        "KL_spec_norm_prod_n_root": "spec_norm_prod", # Plot legends intentionally omit the n-root suffix.
        "KL_path_norm": "path_norm",
        "KL_path_norm_n_root": "path_norm", # Plot legends intentionally omit the n-root suffix.
        "KL_func_det_unsquare_centered": "func_norm",
    }
    for sigma in sigma_indices:
        for alpha in alpha_indices:
            structure_sharpness = f"structureS_sig{sigma}_alpha{alpha}"
            metric_plot_labels[structure_sharpness] = structure_sharpness
    for alpha in input_alpha_indices:
        metric_plot_labels[f"inputS_alpha{alpha}"] = f"inputS_alpha{alpha}"

    # ===============
    SC_metric_pair_labels = {
        pair_name: f"({metric_plot_labels[s_metric]}, {metric_plot_labels[c_metric]})"
        for pair_name, (s_metric, c_metric) in SC_metric_pairs.items()
    }

    # ===============
    # Keep unique original column names in their first-appearance order.
    single_metrics = list(
        dict.fromkeys(
            metric
            for pair in SC_metric_pairs.values()
            for metric in pair
        )
    )

    return SC_metric_pairs, SC_metric_pair_labels, single_metrics


SC_metric_pairs, SC_metric_pair_labels, single_metrics = build_SC_metric_catalog()

# Fail early if a selected metric is unavailable for any model.
required_metric_columns = set(single_metrics)
for model_name, ready_df in metric_dfs_readytogo.items():
    missing_columns = required_metric_columns.difference(ready_df.columns)
    if missing_columns:
        raise KeyError(f"{model_name} is missing metric columns: {sorted(missing_columns)}")

print(
    f"Prepared {len(SC_metric_pairs)} S-C pairs and "
    f"{len(single_metrics)} unique single metrics."
)

display(
    pd.DataFrame(
        {"columns": SC_metric_pairs,
         "plot_label": SC_metric_pair_labels}
    ).rename_axis("pair_name")
)
display(
    pd.DataFrame({"metric": single_metrics})
)

Prepared 23 S-C pairs and 25 unique single metrics.


,columns,plot_label
pair_name,,
pair_original_sqrt,"(S(og), KL(og)_sqrt)","(traceH, l2norm)"
pair_adapsam_spec_norm,"(S_adap_sam, KL_spec_norm_prod)","(adapSAM, spec_norm_prod)"
pair_adapsam_spec_norm_n_root,"(S_adap_sam, KL_spec_norm_prod_n_root)","(adapSAM, spec_norm_prod)"
pair_adapsam_path_norm,"(S_adap_sam, KL_path_norm)","(adapSAM, path_norm)"
pair_adapsam_path_norm_n_root,"(S_adap_sam, KL_path_norm_n_root)","(adapSAM, path_norm)"
pair_adapsam_func_det_unsquare_centered,"(S_adap_sam, KL_func_det_unsquare_centered)","(adapSAM, func_norm)"
pair_structureS_sig1_alpha1_func_det_unsquare_centered,"(structureS_sig1_alpha1, KL_func_det_unsquare_...","(structureS_sig1_alpha1, func_norm)"
pair_structureS_sig1_alpha2_func_det_unsquare_centered,"(structureS_sig1_alpha2, KL_func_det_unsquare_...","(structureS_sig1_alpha2, func_norm)"
pair_structureS_sig1_alpha3_func_det_unsquare_centered,"(structureS_sig1_alpha3, KL_func_det_unsquare_...","(structureS_sig1_alpha3, func_norm)"


,metric
0,S(og)
1,KL(og)_sqrt
2,S_adap_sam
3,KL_spec_norm_prod
4,KL_spec_norm_prod_n_root
5,KL_path_norm
6,KL_path_norm_n_root
7,KL_func_det_unsquare_centered
8,structureS_sig1_alpha1
9,structureS_sig1_alpha2


In [46]:
# IMP

## Step 3 — Regression and Pareto analysis

### Step 3.1 — Analysis helpers

PCR is the fraction of comparable point pairs for which 
the point with no larger S and C has worse target performance. 
Thus, a smaller PCR indicates better agreement between the metric pair and performance.

In [47]:
def filter_well_trained_points(
    df, train_loss_max=None, test_loss_max=None, gap_max=None,
):
    """Return rows satisfying the enabled train/test loss and gap bounds."""
    required = [
        column
        for column, threshold in (
            ("train_loss", train_loss_max),
            ("test_loss", test_loss_max),
            ("gap", gap_max),
        )
        if threshold is not None and column not in df.columns
    ]
    if required:
        raise KeyError(f"Missing filter columns: {required}")

    filtered = df.copy()
    mask = pd.Series(True, index=filtered.index, dtype=bool)
    descriptions = []

    if train_loss_max is not None:
        filtered["train_loss"] = pd.to_numeric(filtered["train_loss"], errors="coerce")
        mask &= filtered["train_loss"].notna()
        mask &= filtered["train_loss"] <= train_loss_max
        descriptions.append(f"train_loss <= {train_loss_max}")

    if test_loss_max is not None:
        filtered["test_loss"] = pd.to_numeric(filtered["test_loss"], errors="coerce")
        mask &= filtered["test_loss"].notna()
        mask &= filtered["test_loss"] <= test_loss_max
        descriptions.append(f"test_loss <= {test_loss_max}")

    if gap_max is not None:
        filtered["gap"] = pd.to_numeric(filtered["gap"], errors="coerce")
        mask &= filtered["gap"].notna()
        mask &= filtered["gap"] <= gap_max
        descriptions.append(f"gap <= {gap_max}")

    filter_description = " and ".join(descriptions) if descriptions else "no filter"
    return filtered.loc[mask].copy(), filter_description


def select_valid_SC_points(df, s_metric, c_metric, target):
    """Keep finite rows with strictly positive S and C values."""
    required = [s_metric, c_metric, target]
    missing = [column for column in required if column not in df.columns]
    if missing:
        raise KeyError(f"Missing analysis columns: {missing}")

    columns = list(dict.fromkeys(["filepath", *required]))
    columns = [column for column in columns if column in df.columns]
    pair_df = df.loc[:, columns].copy()
    for column in required:
        pair_df[column] = pd.to_numeric(pair_df[column], errors="coerce")

    valid_SC_mask = (
        np.isfinite(pair_df[s_metric])
        & np.isfinite(pair_df[c_metric])
        & (pair_df[s_metric] > 0.0)
        & (pair_df[c_metric] > 0.0)
    )
    return pair_df.loc[valid_SC_mask].copy()


def fit_SC_linear_regression(valid_SC_df, s_metric, c_metric, target):
    """Fit 
            target = intercept + beta_S*S + beta_C*C 
        and return 
            R^2 and coefficients.
    """
    regression_df = (
        valid_SC_df[[s_metric, c_metric, target]]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )
    if len(regression_df) < 3:
        return np.nan, None

    X = regression_df[[s_metric, c_metric]]
    y = regression_df[target]

    model = LinearRegression()
    model.fit(X, y)

    r_squared = model.score(X, y)
    coefficient = f"({model.coef_[0]:.4e}, {model.coef_[1]:.4e})"
    return float(r_squared), coefficient


def fit_single_factor_linear_regression(df, metric, target):
    """Fit target = intercept + beta*metric and return fit statistics."""
    required = [metric, target]
    missing = [column for column in required if column not in df.columns]
    if missing:
        raise KeyError(f"Missing single-factor regression columns: {missing}")

    regression_df = df.loc[:, required].copy()
    for column in required:
        regression_df[column] = pd.to_numeric(regression_df[column], errors="coerce")

    regression_df = (
        regression_df
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )
    valid_points = len(regression_df)
    if valid_points < 2:
        return valid_points, np.nan, None

    X = regression_df[[metric]]
    y = regression_df[target]

    model = LinearRegression()
    model.fit(X, y)

    r_squared = model.score(X, y)
    coefficient = f"{model.coef_[0]:.4e}"
    return valid_points, float(r_squared), coefficient


def compute_PCR(
    valid_SC_df, s_metric, c_metric, 
    target, performance_higher_better=False
):
    """Compute the discrepant/comparable pair ratio used as PCR."""
    pcr_df = (
        valid_SC_df[[s_metric, c_metric, target]]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )
    rows = pcr_df.to_numpy(dtype=float)
    comparable_pairs = 0
    discrepant_pairs = 0

    for i in range(len(rows)):
        a_s, a_c, a_performance = rows[i]
        for j in range(i + 1, len(rows)):
            b_s, b_c, b_performance = rows[j]
            a_dominates_b = a_s <= b_s and a_c <= b_c
            b_dominates_a = b_s <= a_s and b_c <= a_c

            if a_dominates_b:
                comparable_pairs += 1
                worse = (
                    a_performance < b_performance
                    if performance_higher_better
                    else a_performance > b_performance
                )
                discrepant_pairs += int(worse)
            elif b_dominates_a:
                comparable_pairs += 1
                worse = (
                    b_performance < a_performance
                    if performance_higher_better
                    else b_performance > a_performance
                )
                discrepant_pairs += int(worse)

    pcr = discrepant_pairs / comparable_pairs if comparable_pairs else np.nan
    return float(pcr) if np.isfinite(pcr) else np.nan


def compute_pareto_front(valid_SC_df, s_metric, c_metric):
    """Return points not dominated when both S and C are minimized."""
    if valid_SC_df.empty:
        return valid_SC_df.copy()

    values = valid_SC_df[[s_metric, c_metric]].to_numpy(dtype=float)
    dominated = np.zeros(len(values), dtype=bool)
    for i, point in enumerate(values):
        weakly_better = np.all(values <= point, axis=1)
        strictly_better = np.any(values < point, axis=1)
        dominated[i] = np.any(weakly_better & strictly_better)

    return (
        valid_SC_df.loc[~dominated]
        .sort_values([s_metric, c_metric])
        .copy()
    )


def parse_SC_pair_label(pair_label):
    """Return the S and C display labels stored in `(S label, C label)`."""
    label_text = str(pair_label).strip()
    if label_text.startswith("(") and label_text.endswith(")"):
        label_text = label_text[1:-1]
    labels = [label.strip() for label in label_text.split(",", maxsplit=1)]
    if len(labels) != 2 or not all(labels):
        raise ValueError(
            f"pair_label must have the form '(S label, C label)', got {pair_label!r}"
        )
    return labels[0], labels[1]


def format_log_tick(value, position=None):
    """Format exact powers of ten as `1e-1`, `1e0`, `1e1`, and so on."""
    if value <= 0.0 or not np.isfinite(value):
        return ""
    exponent = int(np.round(np.log10(value)))
    if not np.isclose(value, 10.0 ** exponent):
        return ""
    return f"1e{exponent}"


def truncate_colormap(cmap_name, minval=0.0, maxval=1.0, n=256):
    """
    Use only a selected portion [minval, maxval] of a Matplotlib colormap.
    """
    cmap = colormaps[cmap_name]

    return LinearSegmentedColormap.from_list(
        f"{cmap_name}_truncated",
        cmap(np.linspace(minval, maxval, n))
    )


def add_slim_colorbar(
    fig, ax, mappable, title, width="2%", pad="2%", tick_labelsize=7,
):
    """Add a slim colorbar to the right and place its title on top."""
    divider = make_axes_locatable(ax)
    colorbar_ax = divider.append_axes("right", size=width, pad=pad)
    colorbar = fig.colorbar(mappable, cax=colorbar_ax)
    colorbar.ax.set_title(title, fontsize=tick_labelsize+1, pad=3)
    colorbar.ax.tick_params(labelsize=tick_labelsize)
    return colorbar


def plot_PCR_pareto(
    valid_SC_df, s_metric, c_metric, target, pair_label, model_name, pcr,
    log_scale=True, figsize=(5,4),
    markersize=10, alpha=0.80, cmap="viridis",
    colorbar_width="2%", colorbar_pad="2%",
    tick_labelsize=8, annotation_fontsize=7,
    show_annotation_arrows=False,
):
    """
    Plot valid S-C points and their lower-left Pareto front.
    """
    plot_df = (
        valid_SC_df.replace([np.inf, -np.inf], np.nan)
        .dropna(subset=[target])
        .copy()
    )
    pareto_df = compute_pareto_front(plot_df, s_metric, c_metric)
    fig, ax = plt.subplots(figsize=figsize)
    x_axis_label, y_axis_label = parse_SC_pair_label(pair_label)
    annotation_texts = []

    if plot_df.empty:
        ax.text(0.5, 0.5, "No valid S-C points", ha="center", va="center")
    else:
        pcr_text = f"{pcr:.1%}" if np.isfinite(pcr) else "N/A"
        custom_cmap = truncate_colormap(
            cmap,
            minval=0.35,
            maxval=0.85,
        )
        scatter = ax.scatter(
            plot_df[s_metric], plot_df[c_metric],
            c=plot_df[target],
            cmap=custom_cmap,
            s=markersize,
            alpha=alpha,
            label=f"{pair_label} PCR={pcr_text}"
        )
        add_slim_colorbar(
            fig, ax, scatter, title=target,
            width=colorbar_width, pad=colorbar_pad,
            tick_labelsize=tick_labelsize,
        )

        # The configured target is `gap`; annotate each point by its value.
        for x_value, y_value, gap_value in zip(
            plot_df[s_metric], plot_df[c_metric], plot_df[target]
        ):
            annotation_texts.append(
                ax.text(
                    x_value, y_value, f"{gap_value:.2f}",
                    fontsize=annotation_fontsize,
                )
            )
        # Match the Pareto line to the selected colormap.
        pareto_line_color = scatter.cmap(0.30)
        ax.plot(
            pareto_df[s_metric], pareto_df[c_metric],
            color=pareto_line_color, linewidth=0.9,
            label="Pareto front",
        )
        ax.legend(fontsize=annotation_fontsize+1.0)

        if log_scale:
            ax.set_xscale("log")
            ax.set_yscale("log")
            # Use plain scientific notation (for example, 1e1) on both axes.
            for axis in (ax.xaxis, ax.yaxis):
                axis.set_major_locator(
                    ticker.LogLocator(base=10.0, subs=(1.0,))
                )
                axis.set_major_formatter(ticker.FuncFormatter(format_log_tick))
                axis.set_minor_formatter(ticker.NullFormatter())

    if annotation_texts:
        adjust_kwargs = {"ax": ax}
        if show_annotation_arrows:
            adjust_kwargs["arrowprops"] = {
                "arrowstyle": "-", "color": "0.4",
                "linewidth": 0.5, "alpha": 0.7,
            }
        adjust_text(annotation_texts, **adjust_kwargs)

    # ax.set_title(f"{model_name}: {pair_label}")
    ax.set_xlabel(f"S metric {x_axis_label}")
    ax.set_ylabel(f"C metric {y_axis_label}")
    ax.tick_params(axis="both", which="both", labelsize=tick_labelsize)
    ax.grid(True, linestyle="--", alpha=0.3)
    fig.tight_layout()
    plt.show()
    return fig, ax


#========== The Engine ==========
def analyze_metric_dataframe(
    metric_df, model_name, SC_metric_pairs, SC_metric_pair_labels,
    single_metrics=None, do_single_factor_reg=True,
    target="gap", train_loss_max=None, test_loss_max=None, gap_max=None,
    performance_higher_better=False,
    sort_summary_table_by=None,
    plot_PCR=False, 
    figsize=(5,4),
    log_scale=True, markersize=10, alpha=0.80,
    cmap="viridis", colorbar_width="5%", colorbar_pad="3%",
    tick_labelsize=9, annotation_fontsize=7,
    show_annotation_arrows=False,
):
    """
    Run filtering, two-factor regression, PCR, optional single-factor
    regression, and optional Pareto plots for one model.
    """
    if target not in metric_df.columns:
        raise KeyError(f"Target column {target!r} is unavailable for {model_name}")
    if sort_summary_table_by not in {None, "PCR", "R^2"}:
        raise ValueError("sort_summary_table_by must be None, 'PCR', or 'R^2'")
    if do_single_factor_reg:
        if single_metrics is None:
            single_metrics = list(
                dict.fromkeys(
                    metric
                    for metric_pair in SC_metric_pairs.values()
                    for metric in metric_pair
                )
            )
        else:
            single_metrics = list(dict.fromkeys(single_metrics))

    well_trained_df, filter_description = filter_well_trained_points(
        metric_df,
        train_loss_max=train_loss_max,
        test_loss_max=test_loss_max,
        gap_max=gap_max,
    )
    print(
        f"{model_name}: {len(well_trained_df)}/{len(metric_df)} well-trained points "
        f"({filter_description})."
    )

    records = []
    valid_data_by_pair = {}
    for pair_name, (s_metric, c_metric) in SC_metric_pairs.items():
        valid_SC_df = select_valid_SC_points(
            well_trained_df, s_metric, c_metric, target
        )
        valid_data_by_pair[pair_name] = valid_SC_df
        # compute PCR
        pcr = compute_PCR(
            valid_SC_df, s_metric, c_metric, target,
            performance_higher_better=performance_higher_better,
        )
        # compute Two-Factor Linear Regression
        r_squared, coefficient = fit_SC_linear_regression(
            valid_SC_df, s_metric, c_metric, target
        )
        records.append(
            {
                "pair_name": pair_name,
                "metric_pair": SC_metric_pair_labels[pair_name],
                "valid_SC_points": len(valid_SC_df),
                "PCR": pcr,
                "R^2": r_squared,
                "coefficient": coefficient,
            }
        )

    summary = pd.DataFrame(records).set_index("pair_name")
    summary = summary[
        ["metric_pair", "valid_SC_points", "PCR", "R^2", "coefficient"]
    ]
    if sort_summary_table_by == "PCR":
        summary = summary.sort_values("PCR", ascending=True, na_position="last")
    elif sort_summary_table_by == "R^2":
        summary = summary.sort_values("R^2", ascending=False, na_position="last")

    single_factor_summary = None
    if do_single_factor_reg:
        single_factor_records = []
        for metric in single_metrics:
            valid_points, r_squared, coefficient = (
                fit_single_factor_linear_regression(
                    well_trained_df, metric, target
                )
            )
            single_factor_records.append(
                {
                    "metric": metric,
                    "valid_points": valid_points,
                    "R^2": r_squared,
                    "coefficient": coefficient,
                }
            )

        single_factor_summary = (
            pd.DataFrame(single_factor_records).set_index("metric")
            [["valid_points", "R^2", "coefficient"]]
        )
        single_factor_summary = single_factor_summary.sort_values(
            "R^2", ascending=False, na_position="last"
        )

    print("\nTwo-factor regression and PCR summary:")
    display(
        summary.style.format(
            {"PCR": "{:.1%}", "R^2": "{:.4f}"}, 
            na_rep="—" # display missing values such as NaN as "-"
        )
    )
    if do_single_factor_reg:
        print("\nSingle-factor regression summary:")
        display(
            single_factor_summary.style.format(
                {"R^2": "{:.4f}"},
                na_rep="—",
            )
        )

    if plot_PCR:
        for pair_name, (s_metric, c_metric) in SC_metric_pairs.items():
            plot_PCR_pareto(
                valid_data_by_pair[pair_name],
                s_metric=s_metric, c_metric=c_metric, target=target,
                pair_label=SC_metric_pair_labels[pair_name],
                model_name=model_name, pcr=summary.loc[pair_name, "PCR"],
                log_scale=log_scale, markersize=markersize, alpha=alpha,
                cmap=cmap, colorbar_width=colorbar_width,
                colorbar_pad=colorbar_pad,
                tick_labelsize=tick_labelsize,
                annotation_fontsize=annotation_fontsize,
                show_annotation_arrows=show_annotation_arrows,
                figsize=figsize,
            )

    return summary, single_factor_summary

### Step 3.2 — Run every model

Set `TRAIN_LOSS_MAX`, `TEST_LOSS_MAX`, and `GAP_MAX` to numbers to enable the corresponding upper bounds. Use `None` to disable any filter. 

Set `PLOT_PCR=True` to show one Pareto plot per metric pair after each model's summary table.

Metric "structureS_alpha1" mathmatically equates "bayesS_aniso".

In [48]:
DATASET_MODELS

{'cifar10': ['resnet18_BN', 'vgg13_BN', 'resnet18_noBN', 'vgg13_noBN'],
 'cifar100': ['resnet34_BN',
  'vgg19_BN',
  'wideresnetpostact_BN',
  'vit_BN',
  'wideresnetpostact_noBN',
  'vit_noBN']}

In [49]:
TRAIN_LOSS_MAX = 0.01
TEST_LOSS_MAX = None  # Example: test_loss <= 2.3 or 4.6
GAP_MAX = 10.0  # Example: gap =test_loss - train_loss


REGRESSION_TARGET = "gap"
PERFORMANCE_HIGHER_BETTER = False
SORT_SUMMARY_TABLE_BY = "PCR"  # None / "PCR" / "R^2"

DO_SINGLE_FACTOR_REG = True

PLOT_PCR = False #>>>>>>>>>>>>>>>>>>.<<<<<<<<<<<<<<<<<<<<<<<<
TICK_LABELSIZE = 8
ANNOTATION_FONTSIZE = 7
SHOW_ANNOTATION_ARROWS = False
PCR_FIGURESIZE =(5, 4.25)
PCR_MARKERSIZE = 8.5
PCR_ALPHA = 0.80
PCR_CMAP = "YlGnBu"  # "viridis" , "plasma", "coolwarm", "magma"
                # Sequential—values increase from low to high:
                # "cividis", "inferno", "Blues", "Greens", "Reds_r", "YlGnBu_r", "BuPu_r"
PCR_COLORBAR_WIDTH = "1.5%"
PCR_COLORBAR_PAD = "2.5%"

regression_pareto_summaries = {}
single_factor_regression_summaries = {}
for model_name in MODELS:
    dataset = MODEL_DATASETS[model_name]
    print(
        f"\n{'=' * 12} Dataset: {dataset} | Model: {model_name} {'=' * 12}"
    )
    
    (
        regression_pareto_summaries[model_name],
        single_factor_regression_summaries[model_name],
    ) = analyze_metric_dataframe(
                    metric_dfs_readytogo[model_name],
                    model_name=model_name,
                    SC_metric_pairs=SC_metric_pairs,
                    SC_metric_pair_labels=SC_metric_pair_labels,
                    single_metrics=single_metrics,
                    do_single_factor_reg=DO_SINGLE_FACTOR_REG,
                    target=REGRESSION_TARGET,
                    train_loss_max=TRAIN_LOSS_MAX,
                    test_loss_max=TEST_LOSS_MAX,
                    gap_max=GAP_MAX,
                    performance_higher_better=PERFORMANCE_HIGHER_BETTER,
                    sort_summary_table_by=SORT_SUMMARY_TABLE_BY,
                    plot_PCR=PLOT_PCR,
                    figsize=PCR_FIGURESIZE,
                    markersize=PCR_MARKERSIZE,
                    alpha=PCR_ALPHA,
                    cmap=PCR_CMAP,
                    colorbar_width=PCR_COLORBAR_WIDTH,
                    colorbar_pad=PCR_COLORBAR_PAD,
                    tick_labelsize=TICK_LABELSIZE,
                    annotation_fontsize=ANNOTATION_FONTSIZE,
                    show_annotation_arrows=SHOW_ANNOTATION_ARROWS,
    )

# Store for further use
resnet18_BN_regression_pareto_summary = regression_pareto_summaries["resnet18_BN"]
vgg13_BN_regression_pareto_summary = regression_pareto_summaries["vgg13_BN"]
resnet18_noBN_regression_pareto_summary = regression_pareto_summaries["resnet18_noBN"]
vgg13_noBN_regression_pareto_summary = regression_pareto_summaries["vgg13_noBN"]
wideresnetpostact_BN_regression_pareto_summary = (regression_pareto_summaries["wideresnetpostact_BN"])
wideresnetpostact_noBN_regression_pareto_summary = (regression_pareto_summaries["wideresnetpostact_noBN"])
vit_BN_regression_pareto_summary = (regression_pareto_summaries["vit_BN"])
vit_noBN_regression_pareto_summary = (regression_pareto_summaries["vit_noBN"])


============ Dataset: cifar10 | Model: resnet18_BN ============
resnet18_BN: 53/72 well-trained points (train_loss <= 0.01 and gap <= 10.0).

Two-factor regression and PCR summary:


,metric_pair,valid_SC_points,PCR,R^2,coefficient
pair_name,,,,,
pair_structureS_sig4_alpha2_func_det_unsquare_centered,"(structureS_sig4_alpha2, func_norm)",53,0.8%,0.8748,"(1.8214e-01, 5.3478e-01)"
pair_structureS_sig5_alpha2_func_det_unsquare_centered,"(structureS_sig5_alpha2, func_norm)",53,0.8%,0.9048,"(3.8802e-02, 5.2613e-01)"
pair_structureS_sig4_alpha1_func_det_unsquare_centered,"(structureS_sig4_alpha1, func_norm)",53,0.8%,0.8697,"(1.7986e-01, 5.3395e-01)"
pair_structureS_sig5_alpha1_func_det_unsquare_centered,"(structureS_sig5_alpha1, func_norm)",53,0.8%,0.9018,"(3.7641e-02, 5.2878e-01)"
pair_structureS_sig1_alpha3_func_det_unsquare_centered,"(structureS_sig1_alpha3, func_norm)",53,1.1%,0.9487,"(4.2963e-01, 3.0101e-01)"
pair_structureS_sig2_alpha3_func_det_unsquare_centered,"(structureS_sig2_alpha3, func_norm)",53,1.1%,0.9566,"(3.9478e-01, 3.2223e-01)"
pair_structureS_sig3_alpha3_func_det_unsquare_centered,"(structureS_sig3_alpha3, func_norm)",53,1.2%,0.9630,"(3.0123e-01, 3.7116e-01)"
pair_structureS_sig5_alpha3_func_det_unsquare_centered,"(structureS_sig5_alpha3, func_norm)",53,1.2%,0.9197,"(4.3005e-02, 4.7206e-01)"
pair_structureS_sig3_alpha2_func_det_unsquare_centered,"(structureS_sig3_alpha2, func_norm)",53,1.2%,0.8346,"(7.9347e-01, 5.2099e-01)"



Single-factor regression summary:


,valid_points,R^2,coefficient
metric,,,
KL_path_norm_n_root,53,0.8457,2.3032e+00
structureS_sig1_alpha3,53,0.7221,5.5081e-01
structureS_sig2_alpha3,53,0.6865,4.9819e-01
structureS_sig3_alpha3,53,0.5801,3.5827e-01
KL_path_norm,53,0.5732,2.5827e-04
KL_func_det_unsquare_centered,53,0.5681,4.4365e-01
KL_spec_norm_prod_n_root,53,0.5441,1.9827e-01
KL(og)_sqrt,53,0.5212,8.6541e-05
inputS_alpha2,53,0.4891,3.8225e-01



============ Dataset: cifar10 | Model: vgg13_BN ============
vgg13_BN: 57/72 well-trained points (train_loss <= 0.01 and gap <= 10.0).

Two-factor regression and PCR summary:


...

,valid_points,R^2,coefficient
metric,,,
KL_func_det_unsquare_centered,47,0.2838,1.0335e+00
inputS_alpha2,47,0.1288,2.9722e-01
inputS_alpha1,47,0.0703,8.4902e-01
KL_spec_norm_prod,47,0.0324,-2.0538e-08
S(og),47,0.0311,5.9926e-05
structureS_sig1_alpha1,47,0.0191,7.1301e+02
structureS_sig1_alpha2,47,0.0136,3.8906e+01
structureS_sig2_alpha2,47,0.0093,1.6957e+01
structureS_sig2_alpha1,47,0.0067,4.7187e+01



============ Dataset: cifar100 | Model: vit_noBN ============
vit_noBN: 28/48 well-trained points (train_loss <= 0.01 and gap <= 10.0).

Two-factor regression and PCR summary:


,metric_pair,valid_SC_points,PCR,R^2,coefficient
pair_name,,,,,
pair_structureS_sig3_alpha3_func_det_unsquare_centered,"(structureS_sig3_alpha3, func_norm)",28,7.8%,0.8577,"(5.4106e-01, 1.8297e+00)"
pair_structureS_sig4_alpha3_func_det_unsquare_centered,"(structureS_sig4_alpha3, func_norm)",28,8.3%,0.8498,"(4.2837e-01, 1.8546e+00)"
pair_structureS_sig1_alpha3_func_det_unsquare_centered,"(structureS_sig1_alpha3, func_norm)",28,8.4%,0.8577,"(5.7870e-01, 1.8374e+00)"
pair_structureS_sig2_alpha3_func_det_unsquare_centered,"(structureS_sig2_alpha3, func_norm)",28,8.4%,0.8574,"(5.6904e-01, 1.8382e+00)"
pair_structureS_sig5_alpha3_func_det_unsquare_centered,"(structureS_sig5_alpha3, func_norm)",28,9.8%,0.7870,"(1.4878e-01, 2.2686e+00)"
pair_structureS_sig5_alpha2_func_det_unsquare_centered,"(structureS_sig5_alpha2, func_norm)",28,11.6%,0.7524,"(1.4016e-01, 2.4234e+00)"
pair_structureS_sig4_alpha2_func_det_unsquare_centered,"(structureS_sig4_alpha2, func_norm)",28,11.9%,0.7863,"(2.6874e+00, 1.9578e+00)"
pair_inputS_alpha2_func_det_unsquare_centered,"(inputS_alpha2, func_norm)",28,12.0%,0.7489,"(1.6212e-01, 2.1961e+00)"
pair_structureS_sig5_alpha1_func_det_unsquare_centered,"(structureS_sig5_alpha1, func_norm)",28,12.0%,0.7480,"(1.3253e-01, 2.4528e+00)"



Single-factor regression summary:


,valid_points,R^2,coefficient
metric,,,
KL_func_det_unsquare_centered,28,0.7011,2.8854e+00
structureS_sig3_alpha3,28,0.6814,8.9253e-01
structureS_sig1_alpha3,28,0.6789,9.5554e-01
structureS_sig2_alpha3,28,0.6785,9.4004e-01
structureS_sig4_alpha3,28,0.6690,7.1800e-01
structureS_sig4_alpha2,28,0.6119,5.2940e+00
structureS_sig4_alpha1,28,0.5388,7.2859e+00
inputS_alpha2,28,0.5278,3.9739e-01
structureS_sig5_alpha3,28,0.4713,2.9749e-01


In [50]:
# IMP

### Step 3.3 — Run every Mixed Model

#### Mixed-model composition

| Combination type | Mixed-model setting | Constituent models |
|---|---|---|
| Cross-architecture C10 | resnet18_vgg13_MixedTwo_ready | resnet18_BN<br>vgg13_BN |
|                        | resnet18_vgg13_MixedFour_ready | resnet18_BN<br>vgg13_BN<br>resnet18_noBN<br>vgg13_noBN |
| Cross-architecture C100| widerenet_vit_MixedTwo_ready | wideresnetpostact_BN<br>vit_BN |
|                        | widerenet_vit_MixedFour_ready | wideresnetpostact_BN<br>wideresnetpostact_noBN<br>vit_BN<br>vit_noBN |
| Cross-architecture and Cross-dataset | resnet18_vgg13_wideresnet_MixedThree_ready | resnet18_BN<br>vgg13_BN<br>wideresnetpostact_BN |
|                                      | resnet18_vgg13_wideresnet_MixedSix_ready | resnet18_BN<br>vgg13_BN<br>wideresnetpostact_BN<br>resnet18_noBN<br>vgg13_noBN<br>wideresnetpostact_noBN |
|                                      | resnet18_vgg13_wideresnet_vit_MixedFour_ready | resnet18_BN<br>vgg13_BN<br>wideresnetpostact_BN<br>vit_BN |
|                                      | resnet18_vgg13_wideresnet_vit_MixedEight_ready | resnet18_BN<br>vgg13_BN<br>wideresnetpostact_BN<br>vit_BN<br>resnet18_noBN<br>vgg13_noBN<br>wideresnetpostact_noBN<br>vit_noBN |

#### Attention‼️
when the mixed model contains ViT,
results involved 'path norm' and 'spec norm prod' are invalid.

In [51]:
mixed_regression_pareto_summaries = {}
mixed_single_factor_regression_summaries = {}
for mixed_name, mixed_df in mixed_metric_dfs_readytogo.items():
    print(f"\n{'=' * 12} Mixed model: {mixed_name} {'=' * 12}")

    (
        mixed_regression_pareto_summaries[mixed_name],
        mixed_single_factor_regression_summaries[mixed_name],
    ) = analyze_metric_dataframe(
        mixed_df,
        model_name=mixed_name,
        SC_metric_pairs=SC_metric_pairs,
        SC_metric_pair_labels=SC_metric_pair_labels,
        single_metrics=single_metrics,
        do_single_factor_reg=DO_SINGLE_FACTOR_REG,
        target=REGRESSION_TARGET,
        train_loss_max=TRAIN_LOSS_MAX,
        test_loss_max=TEST_LOSS_MAX,
        gap_max=GAP_MAX,
        performance_higher_better=PERFORMANCE_HIGHER_BETTER,
        sort_summary_table_by=SORT_SUMMARY_TABLE_BY,
        plot_PCR=PLOT_PCR,
        figsize=PCR_FIGURESIZE,
        markersize=PCR_MARKERSIZE,
        alpha=PCR_ALPHA,
        cmap=PCR_CMAP,
        colorbar_width=PCR_COLORBAR_WIDTH,
        colorbar_pad=PCR_COLORBAR_PAD,
        tick_labelsize=TICK_LABELSIZE,
        annotation_fontsize=ANNOTATION_FONTSIZE,
        show_annotation_arrows=SHOW_ANNOTATION_ARROWS,
    )

# Store for further use
resnet18_vgg13_MixedTwo_regression_pareto_summary = (mixed_regression_pareto_summaries["resnet18_vgg13_MixedTwo_ready"])
resnet18_vgg13_MixedFour_regression_pareto_summary = (mixed_regression_pareto_summaries["resnet18_vgg13_MixedFour_ready"])
widerenet_vit_MixedTwo_regression_pareto_summary = (mixed_regression_pareto_summaries["widerenet_vit_MixedTwo_ready"])
widerenet_vit_MixedFour_regression_pareto_summary = (mixed_regression_pareto_summaries["widerenet_vit_MixedFour_ready"])
resnet18_vgg13_wideresnet_MixedThree_regression_pareto_summary = (mixed_regression_pareto_summaries["resnet18_vgg13_wideresnet_MixedThree_ready"])
resnet18_vgg13_wideresnet_MixedSix_regression_pareto_summary = (mixed_regression_pareto_summaries["resnet18_vgg13_wideresnet_MixedSix_ready"])
resnet18_vgg13_wideresnet_vit_MixedFour_regression_pareto_summary = (mixed_regression_pareto_summaries["resnet18_vgg13_wideresnet_vit_MixedFour_ready"])
resnet18_vgg13_wideresnet_vit_MixedEight_regression_pareto_summary = (mixed_regression_pareto_summaries["resnet18_vgg13_wideresnet_vit_MixedEight_ready"])



============ Mixed model: resnet18_vgg13_MixedTwo_ready ============
resnet18_vgg13_MixedTwo_ready: 110/144 well-trained points (train_loss <= 0.01 and gap <= 10.0).

Two-factor regression and PCR summary:


,metric_pair,valid_SC_points,PCR,R^2,coefficient
pair_name,,,,,
pair_structureS_sig4_alpha3_func_det_unsquare_centered,"(structureS_sig4_alpha3, func_norm)",110,2.6%,0.7531,"(1.7075e-01, 4.2370e-02)"
pair_structureS_sig3_alpha3_func_det_unsquare_centered,"(structureS_sig3_alpha3, func_norm)",110,2.6%,0.7859,"(3.1514e-01, 1.8621e-02)"
pair_structureS_sig5_alpha3_func_det_unsquare_centered,"(structureS_sig5_alpha3, func_norm)",110,2.7%,0.6987,"(4.7560e-02, 7.8216e-02)"
pair_structureS_sig4_alpha1_func_det_unsquare_centered,"(structureS_sig4_alpha1, func_norm)",110,2.7%,0.6174,"(1.4982e-01, 1.0587e-01)"
pair_structureS_sig4_alpha2_func_det_unsquare_centered,"(structureS_sig4_alpha2, func_norm)",110,2.8%,0.6214,"(1.5144e-01, 1.0388e-01)"
pair_structureS_sig5_alpha1_func_det_unsquare_centered,"(structureS_sig5_alpha1, func_norm)",110,2.9%,0.6289,"(3.2584e-02, 9.9046e-02)"
pair_structureS_sig5_alpha2_func_det_unsquare_centered,"(structureS_sig5_alpha2, func_norm)",110,2.9%,0.6334,"(3.4090e-02, 9.7456e-02)"
pair_structureS_sig2_alpha3_func_det_unsquare_centered,"(structureS_sig2_alpha3, func_norm)",110,2.9%,0.8026,"(4.0796e-01, 1.2252e-03)"
pair_structureS_sig1_alpha3_func_det_unsquare_centered,"(structureS_sig1_alpha3, func_norm)",110,3.1%,0.8077,"(4.4389e-01, -5.5132e-03)"



Single-factor regression summary:


,valid_points,R^2,coefficient
metric,,,
structureS_sig1_alpha3,110,0.8073,4.3048e-01
structureS_sig2_alpha3,110,0.8025,4.1077e-01
structureS_sig3_alpha3,110,0.7804,3.5021e-01
inputS_alpha2,110,0.7496,3.2882e-01
KL_path_norm_n_root,110,0.7314,1.4216e+00
structureS_sig4_alpha3,110,0.7175,2.1769e-01
KL_func_det_unsquare_centered,110,0.5251,1.1688e-01
structureS_sig5_alpha3,110,0.5220,7.1487e-02
KL_path_norm,110,0.5035,3.4001e-04



============ Mixed model: resnet18_vgg13_MixedFour_ready ============
resnet18_vgg13_MixedFour_ready: 198/264 well-trained points (train_loss <= 0.01 and gap <= 10.0).

Two-factor regression and PCR summary:


...

,valid_points,R^2,coefficient
metric,,,
KL_path_norm,134,0.6625,1.3877e-04
structureS_sig1_alpha3,181,0.5297,1.1919e+00
structureS_sig2_alpha3,181,0.5091,1.0962e+00
structureS_sig3_alpha3,181,0.4342,8.1890e-01
KL_path_norm_n_root,134,0.3757,1.6102e+00
inputS_alpha2,181,0.3115,6.6613e-01
structureS_sig4_alpha3,181,0.2802,3.6353e-01
inputS_alpha1,181,0.1137,1.9291e+00
structureS_sig5_alpha3,181,0.0925,5.9942e-02



============ Mixed model: resnet18_vgg13_wideresnet_vit_MixedEight_ready ============
resnet18_vgg13_wideresnet_vit_MixedEight_ready: 344/456 well-trained points (train_loss <= 0.01 and gap <= 10.0).

Two-factor regression and PCR summary:


,metric_pair,valid_SC_points,PCR,R^2,coefficient
pair_name,,,,,
pair_structureS_sig5_alpha1_func_det_unsquare_centered,"(structureS_sig5_alpha1, func_norm)",344,7.8%,0.2154,"(3.0855e-02, 4.5355e-01)"
pair_structureS_sig3_alpha3_func_det_unsquare_centered,"(structureS_sig3_alpha3, func_norm)",344,7.9%,0.3841,"(6.2697e-01, 2.7862e-01)"
pair_structureS_sig2_alpha3_func_det_unsquare_centered,"(structureS_sig2_alpha3, func_norm)",344,7.9%,0.4180,"(8.1641e-01, 2.3737e-01)"
pair_structureS_sig5_alpha2_func_det_unsquare_centered,"(structureS_sig5_alpha2, func_norm)",344,8.0%,0.2162,"(3.2238e-02, 4.5218e-01)"
pair_structureS_sig1_alpha3_func_det_unsquare_centered,"(structureS_sig1_alpha3, func_norm)",344,8.0%,0.4272,"(8.7997e-01, 2.2404e-01)"
pair_structureS_sig4_alpha3_func_det_unsquare_centered,"(structureS_sig4_alpha3, func_norm)",344,8.1%,0.3102,"(2.8530e-01, 3.5214e-01)"
pair_structureS_sig4_alpha2_func_det_unsquare_centered,"(structureS_sig4_alpha2, func_norm)",344,8.6%,0.2054,"(8.6707e-02, 4.6590e-01)"
pair_structureS_sig4_alpha1_func_det_unsquare_centered,"(structureS_sig4_alpha1, func_norm)",344,8.6%,0.2043,"(7.9509e-02, 4.6697e-01)"
pair_structureS_sig5_alpha3_func_det_unsquare_centered,"(structureS_sig5_alpha3, func_norm)",344,9.0%,0.2331,"(4.7371e-02, 4.2853e-01)"



Single-factor regression summary:


,valid_points,R^2,coefficient
metric,,,
structureS_sig1_alpha3,344,0.3905,1.0348e+00
structureS_sig2_alpha3,344,0.3762,9.6835e-01
structureS_sig3_alpha3,344,0.3244,7.6518e-01
inputS_alpha2,344,0.2637,6.2463e-01
structureS_sig4_alpha3,344,0.2092,3.7136e-01
KL_func_det_unsquare_centered,344,0.1991,4.6895e-01
structureS_sig5_alpha3,344,0.0738,6.8322e-02
structureS_sig5_alpha2,344,0.0338,4.4979e-02
S_adap_sam,344,0.0326,2.1875e-01


#### Attention‼️
when the mixed model contains ViT,
results involved 'path norm' and 'spec norm prod' are invalid.